In [10]:
from itertools import product
import numpy as np
import networkx as nx
import heapq
import pandas as pd
from tqdm import tqdm 
from collections import deque

In [11]:
def create_problem(size: int, density: float = 1.0, negative_values: bool = False, noise_level: float = 0.0, seed: int = 42):
    rng = np.random.default_rng(seed)
    map_coords = rng.random(size=(size, 2)) # Coordinates
    problem = rng.random((size, size))
    
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            dist = np.sqrt(np.square(map_coords[a, 0] - map_coords[b, 0]) + np.square(map_coords[a, 1] - map_coords[b, 1]))
            problem[a, b] += dist
        else:
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round(), map_coords

In [12]:
def spfa(G, start, goal):
    """
    SPFA: Shortest Path Faster Algorithm
    """
    dist = {node: float('inf') for node in G.nodes}
    dist[start] = 0
    
    parent = {node: None for node in G.nodes}
    
    queue = deque([start]) #Queue FIFO
    in_queue = {start}
    visited_nodes = 0 
    count = {node: 0 for node in G.nodes}
    count[start] = 1
    
    while queue: #until queue is not empty
        current = queue.popleft() #take first node of the queue
        in_queue.remove(current)
        visited_nodes += 1
        
        #relaxation
        for neighbor in G.neighbors(current):
            weight = G[current][neighbor]['weight']
            new_dist = dist[current] + weight
            
            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                parent[neighbor] = current
                
                # add to queue if not present
                if neighbor not in in_queue:
                    queue.append(neighbor)
                    in_queue.add(neighbor)
                    count[neighbor] += 1
                    
                    # detect neg cycle
                    if count[neighbor] > len(G.nodes):
                        return -np.inf, visited_nodes  # neg cycle
    
    # if goal not reachable
    if dist[goal] == float('inf'):
        return np.inf, visited_nodes, []
    
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = parent[curr]
    path.reverse()
    
    return dist[goal], visited_nodes, path

In [13]:
def heuristic(node_idx, target_idx, coords):
    # Euclidean distance
    p1 = coords[node_idx]
    p2 = coords[target_idx]
    return np.sqrt(np.sum((p1 - p2)**2)) * 1000

def astar(G, start, goal, coords):
    """
    A* implementation
    """
    # priority queue: (f_score, g_score, current_node)
    # f = g + h
    h_start = heuristic(start, goal, coords)
    open_set = [(h_start, 0, start)]
    
    # dict of min cost found until now
    g_score = {node: float('inf') for node in G.nodes}
    g_score[start] = 0
    
    came_from={}
    visited_nodes = 0
    
    while open_set:
        _, current_g, current = heapq.heappop(open_set)
        
        if current_g > g_score[current]:
            continue
            
        visited_nodes += 1
        
        if current == goal:
            path=[]
            node = goal
            while node in came_from:
                path.append(node)
                node = came_from[node]
            path.append(start)
            path.reverse()
            return current_g, visited_nodes, path

        for neighbor in G.neighbors(current):
            weight = G[current][neighbor]['weight']
            tentative_g = current_g + weight
            
            if tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                came_from[neighbor] = current
                f_score = tentative_g + heuristic(neighbor, goal, coords)
                heapq.heappush(open_set, (f_score, tentative_g, neighbor))
                
    return np.inf, visited_nodes, []

In [14]:
results = []
sizes = [10, 20, 50, 100, 200] 
densities = [0.2, 0.5, 0.8, 1.0]
noises = [0.0, 0.1, 0.5, 0.8]
negative_values = [False,True] 

NUM_RANDOM_PAIRS = 5 

param_list = list(product(sizes, densities, noises, negative_values))

for size, density, noise, neg in tqdm(param_list):
    problem, coords = create_problem(size, density=density, noise_level=noise, negative_values=neg)
    masked = np.ma.masked_array(problem, mask=np.isinf(problem))
    G = nx.from_numpy_array(masked, create_using=nx.DiGraph)
    
    rng = np.random.default_rng(42) #seed
    
    for _ in range(NUM_RANDOM_PAIRS):
        s = rng.integers(0, size)
        d = rng.integers(0, size)
        while s == d: 
            d = rng.integers(0, size)
        
        # BASELINE 
        try:
            if neg:
                nx_cost = nx.bellman_ford_path_length(G, s, d, weight='weight')
            else:
                nx_cost = nx.shortest_path_length(G, s, d, weight='weight')
        except nx.NetworkXNoPath:
            nx_cost = np.inf
        except nx.NetworkXUnbounded:
            nx_cost = -np.inf 
            
        # A* for positive, spfa for negative
        try:
            if neg:
                my_cost, visited, path = spfa(G, s, d)
                algorithm = "spfa"
            else:
                my_cost, visited, path = astar(G, s, d, coords) 
                algorithm = "astar"
        except Exception as e:
            my_cost = np.nan
            visited = 0
            path = []
            algorithm = "error"
            
        # save only if exists a valid path with positive cost
        if nx_cost != np.inf and nx_cost != -np.inf and nx_cost > 0 and my_cost > 0:
            clean_path = [int(node) for node in path]
            results.append({
                "size": size,
                "density": density,
                "noise": noise,
                "neg": neg,
                "start": s,
                "end": d,
                "baseline_cost": nx_cost,
                "my_cost": my_cost,
                "nodes_visited": visited,
                "error": my_cost - nx_cost,
                "path_length" : len(path),
                "path": str(clean_path),
                "algorithm": algorithm
            })

df = pd.DataFrame(results)
df.to_csv("results.csv", index=False)
print("\nResults in results.csv")

100%|██████████| 160/160 [02:54<00:00,  1.09s/it]


Results in results.csv
